# 02. Electrodermal Activity (EDA) & Motion Artifact Analysis

In wearable biosignal collection, hand movement and ring shifting induce mechanical motion artifacts.
This notebook demonstrates:
1. Loading and aligning synchronous EDA telemetry (`_streamed.csv` / `_computed.csv`) with high-rate accelerometer data (`_imu.csv`).
2. Applying the real-time `SignalConditioner` (Median + 2nd-order Butterworth low-pass filter).
3. Correlating 3-axis accelerometer motion intensity against skin conductance deflections to isolate true Sympathetic Nervous System (SNS) arousal from movement noise.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from nuanic_ring.dsp import SignalConditioner

## 1. Load Session Telemetry

In [ ]:
# Synthetic / Demo data loader (or point to real session logs)
np.random.seed(42)
n_samples = 500
time_s = np.linspace(0, 50, n_samples)

# True skin conductance level (SCL) with occasional skin conductance responses (SCRs)
true_scl = 2.5 + 0.5 * np.sin(2 * np.pi * 0.05 * time_s)
true_scr = 1.2 * np.exp(-((time_s - 25) ** 2) / 4.0)

# Simulated motion spike at t=15s
motion_spike = 4.0 * np.exp(-((time_s - 15) ** 2) / 0.5)

raw_conductance_us = true_scl + true_scr + motion_spike + np.random.normal(0, 0.05, n_samples)
motion_intensity = 1500.0 + 8000.0 * np.exp(-((time_s - 15) ** 2) / 0.5) + np.random.normal(0, 100, n_samples)

df = pd.DataFrame({
    "time_s": time_s,
    "raw_conductance_us": raw_conductance_us,
    "motion_intensity": motion_intensity
})
df.head()

## 2. Real-time Digital Signal Conditioning
We pass raw conductance through `SignalConditioner` with a 1.5 Hz cutoff.

In [ ]:
conditioner = SignalConditioner(sample_rate=10.0, median_kernel=5, cutoff_hz=1.5)
filtered = [conditioner.process(v) for v in df["raw_conductance_us"]]
df["filtered_conductance_us"] = filtered

## 3. Visualize EDA and Motion Alignment
By plotting EDA against IMU motion intensity, we can distinguish the motion artifact at $t=15$s from the physiological SCR peak at $t=25$s.

In [ ]:
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 6), sharex=True)

ax1.plot(df["time_s"], df["raw_conductance_us"], label="Raw Conductance (µS)", alpha=0.5, color="gray")
ax1.plot(df["time_s"], df["filtered_conductance_us"], label="Filtered (Butterworth LPF)", color="blue", lw=2)
ax1.set_ylabel("Skin Conductance (µS)")
ax1.set_title("Electrodermal Activity (EDA) Stream")
ax1.legend()
ax1.grid(True, alpha=0.3)

ax2.plot(df["time_s"], df["motion_intensity"], label="3-Axis IMU Intensity ($x^2+y^2+z^2$)", color="darkorange")
ax2.set_ylabel("IMU Variance")
ax2.set_xlabel("Time (s)")
ax2.set_title("Synchronous Accelerometer Motion Stream")
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()